# Tangram IA — entrenar el detector de fichas por forma

Entrena un YOLOv8-seg que reconoce las 7 fichas **por su forma**, no por su color.

**Por qué hace falta.** El detector actual se entrenó con fichas *sueltas*, una por
foto, y con una clase por color. Medido contra 114 fotos reales de figuras armadas
detecta **1,21 fichas de 7**, y sobre una casa bien armada clasifica las 6 fichas
que ve como `large_tri_orange`. Este entrenamiento ataca las dos causas: las
imágenes traen las 7 fichas **juntas y tocándose**, y el matiz se rota al azar en
cada época para que el color deje de ser una pista.

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno → GPU*.

In [ ]:
!nvidia-smi
# Si esto falla, no hay GPU: ve a Entorno de ejecución → Cambiar tipo de entorno → GPU.

In [ ]:
!pip install -q ultralytics
import ultralytics; ultralytics.checks()

## 1. Subir el código

Sube **`tangram_sintetico.zip`** (unos 16 MB). Lleva el generador, el validador del
proyecto y las 114 superficies reales extraídas de tus fotos.

In [ ]:
from google.colab import files
subido = files.upload()          # elige tangram_sintetico.zip
!unzip -q -o tangram_sintetico.zip -d /content/
%cd /content/tangram_sintetico
!ls sintetico/ | head

## 2. Generar el dataset

6000 imágenes de entrenamiento y 800 de validación. **Tarda unos 25-40 minutos**:
cada imagen se compone y se dibuja desde cero. Si tienes prisa, baja a
`--train 3000 --val 400`; el modelo saldrá algo peor pero el circuito es el mismo.

Las anotaciones son exactas por construcción — se anota el mismo polígono que se
dibuja, así que no hay error de etiquetado posible.

In [ ]:
!python -m sintetico.generar --salida /content/ds --train 6000 --val 800 --lado 640

## 3. Mirar las muestras

**No te saltes esto.** Un dataset puede estar numéricamente perfecto y ser
visualmente inservible, y eso no lo detecta ninguna métrica.

In [ ]:
from IPython.display import Image, display
display(Image('/content/ds/muestras.jpg'))

## 4. Verificar las anotaciones

Comprueba el formato **y** pasa las anotaciones por el validador geométrico del
proyecto: si él ve en cada imagen un Tangram completo, sin fichas montadas ni
sueltas, los datos son coherentes con el sistema que va a consumirlos.

In [ ]:
!python -m sintetico.verificar /content/ds --muestreo 20

## 5. Entrenar

Unas 2-3 horas con GPU. Ultralytics guarda `last.pt` en cada época, así que si
Colab se desconecta no se pierde todo.

`hsv_h=0.5` (dentro de `entrenar.py`) es lo que impide que el modelo vuelva a
apoyarse en el color: rota el matiz por todo el círculo cromático en cada época.
No lo bajes.

In [ ]:
!python -m sintetico.entrenar --dataset /content/ds --epocas 60 --imgsz 640 --batch 16 --device 0

## 6. Resultados del entrenamiento

In [ ]:
from IPython.display import Image, display
display(Image('entrenamientos/tangram_formas/results.png'))
display(Image('entrenamientos/tangram_formas/confusion_matrix_normalized.png'))

## 7. Descargar los pesos

In [ ]:
from google.colab import files
files.download('entrenamientos/tangram_formas/weights/best.pt')

## 8. Y ya en tu PC

1. Copia `best.pt` a `vision-service/models/tangram_formas.pt`
2. En `vision-service/.env`: `YOLO_WEIGHTS=models/tangram_formas.pt`
3. Reinicia el servicio y comprueba en `/health` que `models_loaded` **no** viene vacío.

No hay que tocar el backend ni la app: las clases son las cinco geométricas que
`tangram_validator.resolver_taxonomia()` ya reconoce.

**Después, mide.** Esta es la parte que convierte el trabajo en un resultado
presentable — la comparación contra la línea base sobre tus fotos reales:

```
python evaluar_deteccion.py <carpeta_del_dataset_unet>     --pesos models/tangram_piezas_seg_best.pt     --comparar models/tangram_formas.pt     --splits train,val,test
```

| Modelo actual, medido | |
|---|---|
| IoU de silueta | 0,096 |
| Fichas detectadas | 1,21 de 7 |
| Fotos con las 7 | 2 de 114 |

Cualquier mejora sobre eso será visible. Y ojo: el `val` de este entrenamiento es
sintético, así que un mAP alto ahí **no** demuestra que funcione sobre la mesa.
Lo que lo demuestra es esa tabla.